# RetainIQ — Phase 5.1: Segmentation Dataset & Feature Engineering

## Objective

I prepare a customer-level feature matrix for unsupervised segmentation.

I will segment customers using behavior, value, service adoption, engagement, and account
characteristics. I will not use churn outcomes to create the clusters.

The clustering inputs will exclude:

- `churn_label`
- `churn_category`
- `churn_reason`
- `churn_score`

## 1. Connect to MySQL

In [1]:
from getpass import getpass
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mysql.connector
from mysql.connector import Error

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "retainiq_user",
    "password": getpass("Enter MySQL password for retainiq_user: "),
    "database": "retainiq",
}

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

def run_query(query, params=None):
    connection = None
    cursor = None
    try:
        connection = get_connection()
        cursor = connection.cursor(dictionary=True)
        cursor.execute(query, params or ())
        return pd.DataFrame(cursor.fetchall())
    except Error as exc:
        raise RuntimeError(f"MySQL query failed: {exc}") from exc
    finally:
        if cursor:
            cursor.close()
        if connection and connection.is_connected():
            connection.close()

print("MySQL segmentation connection is ready.")

MySQL segmentation connection is ready.


## 2. Pull the Segmentation Dataset

In [2]:
segmentation_query = """
SELECT
    f.customer_id,
    f.tenure_months,
    f.monthly_charge,
    f.total_charges,
    f.total_refunds,
    f.total_revenue,
    f.satisfaction_score,
    f.cltv,
    f.churn_label,
    f.customer_status,
    d.gender,
    d.age,
    d.senior_citizen,
    d.married,
    d.dependents,
    d.number_of_dependents,
    a.quarter,
    a.referred_a_friend,
    a.number_of_referrals,
    a.offer,
    a.contract,
    a.paperless_billing,
    a.payment_method,
    a.total_extra_data_charges,
    a.total_long_distance_charges,
    s.phone_service,
    s.multiple_lines,
    s.internet_service,
    s.internet_type,
    s.online_security,
    s.online_backup,
    s.device_protection_plan,
    s.premium_tech_support,
    s.streaming_tv,
    s.streaming_movies,
    s.streaming_music,
    s.unlimited_data,
    s.avg_monthly_long_distance_charges,
    s.avg_monthly_gb_download
FROM fact_customer_status f
JOIN dim_demographics d ON f.customer_id = d.customer_id
JOIN dim_account a ON f.customer_id = a.customer_id
JOIN dim_services s ON f.customer_id = s.customer_id;
"""

segmentation_df = run_query(segmentation_query)

print(
    f"Loaded segmentation dataset: "
    f"{len(segmentation_df):,} rows × {segmentation_df.shape[1]:,} columns"
)

Loaded segmentation dataset: 7,043 rows × 39 columns


## 3. Validate Customer Grain

In [3]:
assert len(segmentation_df) == 7043
assert segmentation_df["customer_id"].nunique() == 7043
assert segmentation_df["customer_id"].is_unique

print("PASS — I confirmed one row per customer.")

PASS — I confirmed one row per customer.


## 4. Convert MySQL Numeric Fields

In [4]:
numeric_columns = [
    "tenure_months","monthly_charge","total_charges","total_refunds",
    "total_revenue","satisfaction_score","cltv","age",
    "number_of_dependents","number_of_referrals",
    "total_extra_data_charges","total_long_distance_charges",
    "avg_monthly_long_distance_charges","avg_monthly_gb_download"
]

for column in numeric_columns:
    segmentation_df[column] = pd.to_numeric(
        segmentation_df[column],
        errors="coerce"
    )

print(segmentation_df[numeric_columns].dtypes)

tenure_months                          int64
monthly_charge                       float64
total_charges                        float64
total_refunds                        float64
total_revenue                        float64
satisfaction_score                     int64
cltv                                   int64
age                                    int64
number_of_dependents                   int64
number_of_referrals                    int64
total_extra_data_charges             float64
total_long_distance_charges          float64
avg_monthly_long_distance_charges    float64
avg_monthly_gb_download                int64
dtype: object


## 5. Engineer Service and Engagement Features

In [5]:
service_columns = [
    "phone_service",
    "online_security",
    "online_backup",
    "device_protection_plan",
    "premium_tech_support",
    "streaming_tv",
    "streaming_movies",
    "streaming_music",
    "unlimited_data"
]

for column in service_columns:
    segmentation_df[f"{column}_flag"] = (
        segmentation_df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yes")
        .astype(int)
    )

segmentation_df["service_count"] = segmentation_df[
    [f"{column}_flag" for column in service_columns]
].sum(axis=1)

segmentation_df["referred_friend_flag"] = (
    segmentation_df["referred_a_friend"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
    .astype(int)
)

segmentation_df["paperless_billing_flag"] = (
    segmentation_df["paperless_billing"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
    .astype(int)
)

segmentation_df[[
    "customer_id",
    "service_count",
    "referred_friend_flag",
    "paperless_billing_flag"
]].head()

,customer_id,service_count,referred_friend_flag,paperless_billing_flag
0,0002-ORFBO,5,1,1
1,0003-MKNFE,3,0,0
2,0004-TLHLJ,3,0,1
3,0011-IGKFF,6,1,1
4,0013-EXCHZ,4,1,1


## 6. Build the Clustering Feature Matrix

In [6]:
numeric_features = [
    "tenure_months",
    "monthly_charge",
    "total_revenue",
    "cltv",
    "satisfaction_score",
    "avg_monthly_gb_download",
    "avg_monthly_long_distance_charges",
    "number_of_referrals",
    "number_of_dependents",
    "total_extra_data_charges",
    "total_long_distance_charges",
    "service_count",
    "referred_friend_flag",
    "paperless_billing_flag"
]

categorical_features = [
    "contract",
    "internet_type",
    "payment_method",
    "offer"
]

feature_df = segmentation_df[
    ["customer_id"] + numeric_features + categorical_features
].copy()

for column in numeric_features:
    feature_df[column] = feature_df[column].fillna(
        feature_df[column].median()
    )

for column in categorical_features:
    feature_df[column] = feature_df[column].fillna("Unknown").astype(str)

feature_matrix = pd.get_dummies(
    feature_df.drop(columns=["customer_id"]),
    columns=categorical_features,
    dtype=float
)

print(
    f"Final clustering matrix: "
    f"{feature_matrix.shape[0]:,} rows × {feature_matrix.shape[1]:,} features"
)

Final clustering matrix: 7,043 rows × 30 features


## 7. Standardize the Feature Matrix

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_matrix)

print(f"Scaled matrix shape: {X_scaled.shape}")

Scaled matrix shape: (7043, 30)


## 8. Segmentation Design

```text
Customer characteristics
          ↓
     Feature engineering
          ↓
       K-Means
          ↓
    Customer segments
          ↓
 Churn / value profiling
```

The clusters are created independently of the churn outcome.

# Phase 5.1 Conclusion

I created the segmentation dataset directly from MySQL, engineered service and engagement
features, encoded categorical variables, handled missing values, and standardized the feature
matrix.

**Next:** evaluate candidate cluster structures.